In [19]:
import pandas as pd

import sys
import os

project_root = os.path.abspath("..")

if project_root not in sys.path:
    sys.path.append(project_root)

# Load dataset
df = pd.read_excel("../data/raw/routes_performance.xlsx")

# Create empty list to store processed routes
route_dataset = []

# Group by Route ID
grouped_routes = df.groupby("Route ID")

# Process each route
for route_id, route_df in grouped_routes:

    # -----------------------------
    # Planned Route
    # -----------------------------
    planned_route_df = route_df.sort_values("IndexP")

    planned_route = planned_route_df["Address ID"].tolist()

    # -----------------------------
    # Actual Route
    # -----------------------------
    actual_route_df = route_df.sort_values("IndexA")

    actual_route = actual_route_df["Address ID"].tolist()

    # -----------------------------
    # Store in dataset
    # -----------------------------

    driver_id = route_df["Driver ID"].iloc[0]

    country = route_df["Country"].iloc[0]

    week_id = route_df["Week ID"].iloc[0]

    day_of_week = route_df["Day of Week"].iloc[0]


    planned_total_distance = (
    route_df["DistanceP"].sum()
    )

    actual_total_distance = (
        route_df["DistanceA"].sum()
    )
    route_dataset.append({

    "Route ID": route_id,

    "Driver ID": driver_id,

    "Country": country,

    "Week ID": week_id,

    "Day of Week": day_of_week,

    "Planned Route": planned_route,

    "Actual Route": actual_route,

    "Planned Stop Count":
        len(planned_route),

    "Actual Stop Count":
        len(actual_route),

    "Planned Distance":
        planned_total_distance,

    "Actual Distance":
        actual_total_distance
})

# Convert to DataFrame
route_dataset_df = pd.DataFrame(route_dataset)

# Show first 5 rows
print(route_dataset_df.head())

# Optional: dataset shape
print("\nDataset Shape:")
print(route_dataset_df.shape)

print(df[df["Route ID"] == 0][["Stop ID", "Address ID", "IndexP", "IndexA"]])

print(route_dataset_df.iloc[0]["Planned Route"])
print(route_dataset_df.iloc[0]["Actual Route"])

from src.metrics.rqs import calculate_rqs
from src.metrics.eds import calculate_eds
from src.metrics.dds import calculate_dds


route_dataset_df["RQS"] = route_dataset_df.apply(
    lambda row: calculate_rqs(
        row["Planned Route"],
        row["Actual Route"]
    ),
    axis=1
)

route_dataset_df["EDS"] = route_dataset_df.apply(
    lambda row: calculate_eds(
        row["Planned Route"],
        row["Actual Route"]
    ),
    axis=1
)

from src.metrics.dds import calculate_dds

route_dataset_df["DDS"] = route_dataset_df.apply(
    lambda row: calculate_dds(
        row["Planned Distance"],
        row["Actual Distance"]
    ),
    axis=1
)

print(
    route_dataset_df[
        ["RQS", "EDS", "DDS"]
    ].describe()
)

print(route_dataset_df.head())


print(
    route_dataset_df.sort_values(
        "RQS"
    ).head(5)
)
print(
    route_dataset_df.sort_values(
        "EDS"
    ).head(5)
)

print(
    route_dataset_df.sort_values(
        "DDS"
    ).head(5)
)

print()

print(
    route_dataset_df.sort_values(
        "DDS",
        ascending=False
    ).head(5)
)


from src.preprocessing.label_generator import generate_label

route_dataset_df["Label"] = route_dataset_df.apply(
    lambda row: generate_label(
        row["RQS"],
        row["EDS"],
        row["DDS"]
    ),
    axis=1
)

print(route_dataset_df["Label"].value_counts())

# --------------------------------
# Save processed dataset
# --------------------------------

route_dataset_df.to_pickle(
    "../data/processed/route_dataset.pkl"
)

route_dataset_df.to_csv(
    "../data/processed/route_dataset.csv",
    index=False
)

print("Dataset saved successfully!")

route_dataset_df.head()

   Route ID  Driver ID  Country  Week ID Day of Week  \
0         0          0        1        0      Monday   
1         1          1        1        0      Monday   
2         2          2        1        0      Monday   
3         3          3        1        0      Monday   
4         4          4        1        0      Monday   

                             Planned Route  \
0                    [0, 1, 2, 3, 3, 0, 1]   
1                    [0, 4, 5, 6, 7, 8, 9]   
2              [0, 10, 11, 12, 13, 14, 15]   
3  [0, 16, 17, 18, 19, 20, 21, 22, 20, 23]   
4          [0, 24, 25, 26, 27, 28, 29, 30]   

                              Actual Route  Planned Stop Count  \
0                    [0, 3, 3, 0, 1, 2, 1]                   7   
1                    [0, 9, 4, 7, 6, 5, 8]                   7   
2              [0, 12, 13, 11, 10, 14, 15]                   7   
3  [0, 16, 19, 17, 18, 20, 22, 20, 21, 23]                  10   
4          [0, 24, 30, 26, 27, 28, 29, 25]              

,Route ID,Driver ID,Country,Week ID,Day of Week,Planned Route,Actual Route,Planned Stop Count,Actual Stop Count,Planned Distance,Actual Distance,RQS,EDS,DDS,Label
0,0,0,1,0,Monday,"[0, 1, 2, 3, 3, 0, 1]","[0, 3, 3, 0, 1, 2, 1]",7,7,49.468094,44.965197,0.5000,0.5714,-0.0910,D
1,1,1,1,0,Monday,"[0, 4, 5, 6, 7, 8, 9]","[0, 9, 4, 7, 6, 5, 8]",7,7,33.274342,33.610418,0.5000,0.5714,0.0101,D
2,2,2,1,0,Monday,"[0, 10, 11, 12, 13, 14, 15]","[0, 12, 13, 11, 10, 14, 15]",7,7,12.124804,12.508786,0.6667,0.5714,0.0317,D
3,3,3,1,0,Monday,"[0, 16, 17, 18, 19, 20, 21, 22, 20, 23]","[0, 16, 19, 17, 18, 20, 22, 20, 21, 23]",10,10,19.039848,19.374644,0.8400,0.4000,0.0176,D
4,4,4,1,0,Monday,"[0, 24, 25, 26, 27, 28, 29, 30]","[0, 24, 30, 26, 27, 28, 29, 25]",8,8,20.632674,19.528799,0.6875,0.2500,-0.0535,D


In [3]:
from src.metrics.eds import calculate_eds

route_dataset_df["EDS"] = route_dataset_df.apply(
    lambda row: calculate_eds(
        row["Planned Route"],
        row["Actual Route"]
    ),
    axis=1
)

print(route_dataset_df[["RQS", "EDS"]].describe())

                RQS           EDS
count  19647.000000  19647.000000
mean       0.821134      0.325455
std        0.189716      0.261859
min        0.000000      0.000000
25%        0.714300      0.000000
50%        0.875000      0.333300
75%        1.000000      0.538500
max        1.000000      1.000000


In [11]:
from src.preprocessing.label_generator import generate_label

route_dataset_df["Label"] = route_dataset_df.apply(
    lambda row:
        generate_label(
            row["RQS"],
            row["EDS"],
            row["DDS"]
        ),
    axis=1
)

print(
    route_dataset_df["Label"]
    .value_counts()
)

Label
D     12158
ND     7489
Name: count, dtype: int64


In [13]:
route_dataset_df["Label"] = (
    (route_dataset_df["RQS"] >= 0.85) &
    (route_dataset_df["EDS"] <= 0.30) &
    (route_dataset_df["DDS"] <= 0.03)
)

route_dataset_df["Label"] = route_dataset_df["Label"].map(
    {
        True: "ND",
        False: "D"
    }
)

print(route_dataset_df["Label"].value_counts())

print(
    "\nPercentage:\n",
    round(
        route_dataset_df["Label"]
        .value_counts(normalize=True) * 100,
        2
    )
)

print(
    route_dataset_df[
        [
            "RQS",
            "EDS",
            "DDS",
            "Label"
        ]
    ].head(10)
)

Label
D     11505
ND     8142
Name: count, dtype: int64

Percentage:
 Label
D     58.56
ND    41.44
Name: proportion, dtype: float64
      RQS     EDS     DDS Label
0  0.5000  0.5714 -0.0910     D
1  0.5000  0.5714  0.0101     D
2  0.6667  0.5714  0.0317     D
3  0.8400  0.4000  0.0176     D
4  0.6875  0.2500 -0.0535     D
5  0.2000  0.8000 -0.1820     D
6  0.6735  0.5714 -0.0097     D
7  0.6964  0.7333  0.1843     D
8  0.9167  0.2857 -0.0353    ND
9  0.2000  0.7273 -0.2616     D


In [15]:
from src.preprocessing.label_generator import generate_label

route_dataset_df["Label"] = route_dataset_df.apply(
    lambda row: generate_label(
        row["RQS"],
        row["EDS"],
        row["DDS"]
    ),
    axis=1
)

print(route_dataset_df["Label"].value_counts())

print(
    round(
        route_dataset_df["Label"]
        .value_counts(normalize=True) * 100,
        2
    )
)

print(
    route_dataset_df[
        ["RQS", "EDS", "DDS", "Label"]
    ].head(10)
)

Label
D     12158
ND     7489
Name: count, dtype: int64
Label
D     61.88
ND    38.12
Name: proportion, dtype: float64
      RQS     EDS     DDS Label
0  0.5000  0.5714 -0.0910     D
1  0.5000  0.5714  0.0101     D
2  0.6667  0.5714  0.0317     D
3  0.8400  0.4000  0.0176     D
4  0.6875  0.2500 -0.0535     D
5  0.2000  0.8000 -0.1820     D
6  0.6735  0.5714 -0.0097     D
7  0.6964  0.7333  0.1843     D
8  0.9167  0.2857 -0.0353     D
9  0.2000  0.7273 -0.2616     D


In [16]:
route_dataset_df.head()

,Route ID,Planned Route,Actual Route,Planned Stop Count,Actual Stop Count,Planned Distance,Actual Distance,RQS,EDS,DDS,Label
0,0,"[0, 1, 2, 3, 3, 0, 1]","[0, 3, 3, 0, 1, 2, 1]",7,7,49.468094,44.965197,0.5000,0.5714,-0.0910,D
1,1,"[0, 4, 5, 6, 7, 8, 9]","[0, 9, 4, 7, 6, 5, 8]",7,7,33.274342,33.610418,0.5000,0.5714,0.0101,D
2,2,"[0, 10, 11, 12, 13, 14, 15]","[0, 12, 13, 11, 10, 14, 15]",7,7,12.124804,12.508786,0.6667,0.5714,0.0317,D
3,3,"[0, 16, 17, 18, 19, 20, 21, 22, 20, 23]","[0, 16, 19, 17, 18, 20, 22, 20, 21, 23]",10,10,19.039848,19.374644,0.8400,0.4000,0.0176,D
4,4,"[0, 24, 25, 26, 27, 28, 29, 30]","[0, 24, 30, 26, 27, 28, 29, 25]",8,8,20.632674,19.528799,0.6875,0.2500,-0.0535,D
